# 🎓 Student Placement Prediction — Advanced Classification Portfolio Project

---

| Field | Details |
|---|---|
| **Dataset** | Campus Recruitment dataset — `Placement_Data_Full_Class.csv` (Kaggle) |
| **Type** | Supervised Learning · Binary Classification |
| **Level** | Advanced · Portfolio Grade |
| **Models** | Logistic Regression · KNN · Naive Bayes · SVM · Random Forest · Gradient Boosting |
| **Advanced** | ROC-AUC · Precision-Recall Curves · SMOTE · class_weight · GridSearchCV · Learning Curves · Probability Calibration · Feature Importance |

---

## 📌 Project Objective

Predict whether a student will be **placed (hired)** or **not placed** based on academic records and test scores.

This is a **binary classification** problem with real-world stakes:
- A college wants to identify **at-risk students** early and provide targeted support
- An accurate model can guide resource allocation for placement training

**Kaggle dataset challenges:**
- Real data with missing values and mixed data types
- Moderate class imbalance
- Categorical features requiring encoding
- Feature selection from a meaningful but limited feature set


## Section 01

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import warnings 
warnings.filterwarnings('ignore')

from scipy import stats

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    GridSearchCV, learning_curve
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report, average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# SMOTE for Class imbalance

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_AVAILABLE = True
    print("Imbalanced-learn available")

except ImportError:
    SMOTE_AVAILABLE = False
    print("imbalanced-learn not installed.")

RANDOM_STATE = 42

plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')
print('All Library Loaded. Lets get started...')


Imbalanced-learn available
All Library Loaded. Lets get started...


## Section 2 - Data Loading 

**Dataset:** Download `Placement_Data_Full_Class.csv` from
[Kaggle — Campus Recruitment](https://www.kaggle.com/benroshan/factors-affecting-campus-placement)

| Column | Description |
|---|---|
| `sl_no` | Serial number (drop — not a feature) |
| `gender` | Male / Female |
| `ssc_p` | Secondary school % (10th grade) |
| `ssc_b` | Secondary school board (Central / Others) |
| `hsc_p` | Higher secondary % (12th grade) |
| `hsc_b` | Higher secondary board |
| `hsc_s` | Higher secondary stream (Science/Commerce/Arts) |
| `degree_p` | Degree percentage |
| `degree_t` | Degree type |
| `workex` | Work experience (Yes/No) |
| `etest_p` | Employability test percentage |
| `specialisation` | MBA specialisation (Mkt&HR / Mkt&Fin) |
| `mba_p` | MBA percentage |
| `status` | **Target** — Placed / Not Placed |
| `salary` | Salary offered (only if placed — we will drop this to avoid leakage) |

In [5]:
df = pd.read_csv('Data\Placement_Data_Full_Class.csv')

print(f"Dataset Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nTarget Distribution: ")
print(df['status'].value_counts())
print(f"\nPlacement Rate: {(df["status"] == "Placed").mean()*100:.1f}%")

df.head()


Dataset Loaded: 215 rows x 15 columns

Columns: ['sl_no', 'gender', 'ssc_p', 'ssc_b', 'hsc_p', 'hsc_b', 'hsc_s', 'degree_p', 'degree_t', 'workex', 'etest_p', 'specialisation', 'mba_p', 'status', 'salary']

Target Distribution: 
status
Placed        148
Not Placed     67
Name: count, dtype: int64

Placement Rate: 68.8%


,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,1,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed,270000.0
1,2,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed,200000.0
2,3,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed,250000.0
3,4,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed,NaN
4,5,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed,425000.0


## Section 3 - Exploratory Data Analysis (EDA)

In [12]:
# 3.1 Overview
